# COMP9517 Group Project — Part A: Dataset & Data Pipeline

## 1. Setup

In [1]:
import os
for d in ["data/raw", "data/metadata", "data/subset", "scripts", "src/data"]:
    os.makedirs(d, exist_ok=True)
print("Project folders ready.")

Project folders ready.


### 2.Sanity-check the annotation file schema

Run this once you have the real `data/raw/train_mini.json` — confirms
the field names match what `build_dataset.py` expects (`id`, `name`,
`supercategory`/`kingdom` for categories; `id`, `file_name` for images).
If the printed keys differ, adjust `load_annotation_file()` in
`scripts/build_dataset.py` accordingly.


In [2]:
import json, os
train_json_path = "data/raw/train_mini.json"
if os.path.exists(train_json_path):
    d = json.load(open(train_json_path))
    print("category sample:", d["categories"][0])
    print("image sample:", d["images"][0])
else:
    print(f"{train_json_path} not found yet — download the data first.")


category sample: {'id': 0, 'name': 'Lumbricus terrestris', 'common_name': 'Common Earthworm', 'supercategory': 'Animalia', 'kingdom': 'Animalia', 'phylum': 'Annelida', 'class': 'Clitellata', 'order': 'Haplotaxida', 'family': 'Lumbricidae', 'genus': 'Lumbricus', 'specific_epithet': 'terrestris', 'image_dir_name': '00000_Animalia_Annelida_Clitellata_Haplotaxida_Lumbricidae_Lumbricus_terrestris'}
image sample: {'id': 0, 'width': 500, 'height': 500, 'file_name': 'train_mini/02912_Animalia_Chordata_Actinopterygii_Siluriformes_Ictaluridae_Ameiurus_nebulosus/d615f184-8af4-4c60-b9f8-3081c1607644.jpg', 'license': 0, 'rights_holder': 'Ken-ichi Ueda', 'date': '2010-07-14 20:19:00+00:00', 'latitude': 43.83486, 'longitude': -71.22231, 'location_uncertainty': 77}


## 3. Unified data interface: `src/data/`

These four files are what B/C/D import from — do not let them re-parse
raw JSON themselves; everyone reads through this interface.


In [3]:
%%writefile src/data/transforms.py
"""
src/data/transforms.py
------------------------
Unified image preprocessing (Section 4 of the spec).

Two interfaces:
    get_raw_transform()          -> for B (traditional CV): resize only, NO
                                     ImageNet normalisation (B works on raw
                                     pixel/gradient features, not CNN features)
    get_cnn_transform(transform_type) -> for C/D: 224x224, RGB, ImageNet
                                     mean/std, with none/basic_aug/strong_aug
                                     augmentation levels for C's ablation study
"""

from torchvision import transforms

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_raw_transform(resize=True):
    """For B (traditional CV pipeline). Resize only, no normalisation --
    B extracts SIFT/HOG features from raw pixel values."""
    ops = []
    if resize:
        ops.append(transforms.Resize((IMG_SIZE, IMG_SIZE)))
    return transforms.Compose(ops) if ops else None


def get_cnn_transform(transform_type: str):
    """For C/D (CNN pipelines). transform_type: 'none' / 'basic_aug' / 'strong_aug'."""
    normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

    if transform_type == "none":
        # No augmentation -- used for val/test, and for C's "No Aug" ablation arm
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            normalize,
        ])

    elif transform_type == "basic_aug":
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            normalize,
        ])

    elif transform_type == "strong_aug":
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            normalize,
            transforms.RandomErasing(p=0.25),
        ])

    else:
        raise ValueError(f"Unknown transform_type: {transform_type}. "
                          f"Must be 'none', 'basic_aug', or 'strong_aug'.")


Overwriting src/data/transforms.py


In [4]:
%%writefile src/data/dataset.py
"""
src/data/dataset.py
---------------------
Core Dataset class. Everyone (A/B/C/D) reads data through this -- nobody
re-splits or re-reads raw JSON themselves (data合同 rule #1).

Usage (B, raw interface, Section 4.1):
    dataset = INatDataset(csv_file="data/metadata/train.csv", transform=None)
    image, class_idx, image_id = dataset[0]   # image is a PIL.Image if transform=None

Usage (C/D, via get_dataloader in dataloader.py -- don't use this directly):
    dataset = INatDataset(csv_file="data/metadata/train.csv",
                           transform=get_cnn_transform("basic_aug"))
"""

import csv
from PIL import Image
from torch.utils.data import Dataset


class INatDataset(Dataset):
    def __init__(self, csv_file: str, transform=None, csv_root: str = "."):
        """
        Args:
            csv_file: path to train.csv / val.csv / test.csv / longtail_train.csv
                      (must follow the exact schema from Section 3.1:
                      image_id,image_path,original_class_id,class_idx,class_name,split)
            transform: any torchvision transform, or None to get raw PIL images
                       (B's use case -- resize can still be applied via transform
                       if desired, just skip ImageNet normalisation).
            csv_root: prefix to join with each row's image_path, in case the CSV
                      stores paths relative to the project root but you're running
                      from a different working directory.
        """
        self.rows = self._load_csv(csv_file)
        self.transform = transform
        self.csv_root = csv_root

    @staticmethod
    def _load_csv(csv_file):
        rows = []
        with open(csv_file, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                rows.append({
                    "image_id": row["image_id"],
                    "image_path": row["image_path"],
                    "original_class_id": int(row["original_class_id"]),
                    "class_idx": int(row["class_idx"]),
                    "class_name": row["class_name"],
                    "split": row["split"],
                })
        return rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        image_path = f"{self.csv_root}/{row['image_path']}" if self.csv_root != "." else row["image_path"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, row["class_idx"], row["image_id"]

    def get_class_counts(self):
        """Returns {class_idx: count} -- useful for A's longtail construction
        and for building a WeightedRandomSampler (see sampling.py)."""
        counts = {}
        for row in self.rows:
            counts[row["class_idx"]] = counts.get(row["class_idx"], 0) + 1
        return counts


Overwriting src/data/dataset.py


In [5]:
%%writefile src/data/sampling.py
"""
src/data/sampling.py
-----------------------
Utilities for Section 9 (long-tail / class imbalance experiments).

get_weighted_sampler(): implements "方式B（动态采样，推荐）" from the spec --
inverse-class-frequency WeightedRandomSampler, used at train time. No CSV
duplication needed; just pass use_weighted_sampler=True to get_dataloader().
"""

from torch.utils.data import WeightedRandomSampler


def get_weighted_sampler(dataset) -> WeightedRandomSampler:
    """
    Builds a WeightedRandomSampler that gives each SAMPLE a weight of
    1 / (count of its class), so minority classes are drawn roughly as
    often as majority classes over the course of an epoch.

    Args:
        dataset: an INatDataset instance (or anything with .rows[i]["class_idx"]
                 and .get_class_counts()).
    """
    class_counts = dataset.get_class_counts()  # {class_idx: count}
    weights = [1.0 / class_counts[row["class_idx"]] for row in dataset.rows]
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)


Overwriting src/data/sampling.py


In [6]:
%%writefile src/data/dataloader.py
"""
src/data/dataloader.py
------------------------
The real get_dataloader(), matching Section 4.2 of the spec exactly.

    train_loader = get_dataloader(
        split="train",
        transform_type="basic_aug",   # none / basic_aug / strong_aug
        batch_size=64
    )

C/D: once this file exists, delete src/data/dataloader_stub.py and change
your imports from
    from src.data.dataloader_stub import get_dataloader
to
    from src.data.dataloader import get_dataloader
No other code changes needed -- same signature, same (image, class_idx,
image_id) return per sample.

Extra optional args (csv_override, use_weighted_sampler) support A's
longtail experiments (Section 9) without breaking the standard signature
for everyone else -- they're keyword-only with safe defaults.
"""

from torch.utils.data import DataLoader

from src.data.dataset import INatDataset
from src.data.transforms import get_cnn_transform
from src.data.sampling import get_weighted_sampler

METADATA_DIR = "data/metadata"


def get_dataloader(split: str, transform_type: str = "basic_aug", batch_size: int = 64,
                    *, csv_override: str = None, use_weighted_sampler: bool = False,
                    num_workers: int = 2):
    """
    Args:
        split: "train" / "val" / "test"
        transform_type: "none" / "basic_aug" / "strong_aug"
        batch_size: batch size
        csv_override: use a different CSV than the standard split file, e.g.
                       "data/metadata/longtail_train.csv" for A's longtail
                       experiment (Section 9). Only intended for split="train".
        use_weighted_sampler: if True, uses a WeightedRandomSampler based on
                       inverse class frequency (for longtail experiments).
                       Cannot be combined with shuffle=True (mutually exclusive
                       in PyTorch), handled automatically below.
        num_workers: DataLoader worker processes.
    """
    if split not in {"train", "val", "test"}:
        raise ValueError(f"Unknown split: {split}")

    csv_file = csv_override or f"{METADATA_DIR}/{split}.csv"
    transform = get_cnn_transform(transform_type)
    dataset = INatDataset(csv_file=csv_file, transform=transform)

    # Per 全组数据合同 rule #5: val/test always shuffle=False so predictions.csv /
    # scores.npz alignment by image_id is unambiguous.
    if split != "train":
        return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    if use_weighted_sampler:
        sampler = get_weighted_sampler(dataset)
        return DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=num_workers)

    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)


Overwriting src/data/dataloader.py


## 4. Build the 500-class split + metadata

Writes and then runs `scripts/build_dataset.py`. Produces everything in
`data/metadata/`: `selected_classes.csv`, `class_to_idx.json`,
`idx_to_class.json`, `train.csv`, `val.csv`, `test.csv`,
`split_config.json`.

🔴 Uses `--seed 500` — the whole group must use this same seed. Don't
re-run with a different seed after the team has started training.


In [7]:
%%writefile scripts/build_dataset.py
"""
scripts/build_dataset.py
--------------------------
A's core deliverable. Reads the official iNat2021 mini annotation files
and produces every metadata file listed in Section 3 of the spec:

    data/metadata/selected_classes.csv
    data/metadata/class_to_idx.json
    data/metadata/idx_to_class.json
    data/metadata/train.csv
    data/metadata/val.csv
    data/metadata/test.csv
    data/metadata/split_config.json

Usage:
    python scripts/build_dataset.py \
        --train_json data/raw/train_mini.json \
        --val_json data/raw/val.json \
        --num_classes 500 \
        --seed 500 \
        --output_dir data/metadata

NOTE ON JSON SCHEMA: iNat's COCO-style annotation files have this shape
(confirmed against the 2019/2021 releases -- double check field names
against your actual downloaded train_mini.json once you have it, since
Anthropic's sandbox couldn't download the real 42GB file to verify):

    {
      "images": [{"id": int, "file_name": str, "width": int, "height": int, ...}],
      "annotations": [{"id": int, "image_id": int, "category_id": int}],
      "categories": [{"id": int, "name": str, "supercategory": str,
                       "kingdom": str, "phylum": str, "class": str,
                       "order": str, "family": str, "genus": str}]
    }

If your actual file uses slightly different key names, adjust the
`.get(...)` calls in `load_annotation_file()` below -- everything else
in this script is schema-agnostic once it has (image_id -> file_name)
and (image_id -> category_id) mappings.
"""

import os
import csv
import json
import random
import argparse
from collections import defaultdict


def load_annotation_file(json_path):
    """Returns (images: {image_id: file_name}, image_to_category: {image_id: category_id},
    categories: {category_id: {"name":..., "supercategory":...}})."""
    with open(json_path) as f:
        data = json.load(f)

    images = {img["id"]: img["file_name"] for img in data["images"]}
    image_to_category = {ann["image_id"]: ann["category_id"] for ann in data["annotations"]}
    categories = {
        cat["id"]: {
            "name": cat.get("name", f"category_{cat['id']}"),
            # supercategory is the coarse group (Aves/Plantae/Insecta/...); fall back to kingdom
            "category": cat.get("supercategory") or cat.get("kingdom") or "unknown",
        }
        for cat in data["categories"]
    }
    return images, image_to_category, categories


def group_images_by_category(images, image_to_category):
    """Returns {category_id: [image_id, ...]}"""
    grouped = defaultdict(list)
    for image_id, category_id in image_to_category.items():
        if image_id in images:
            grouped[category_id].append(image_id)
    return grouped


def normalize_path(raw_root, file_name):
    """file_name from the json is usually already a relative path
    like 'train_mini/00123/abc.jpg'. Make sure it's prefixed with data/raw/."""
    file_name = file_name.lstrip("/")
    if file_name.startswith("data/raw/"):
        return file_name
    return os.path.join(raw_root, file_name).replace("\\", "/")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_json", default="data/raw/train_mini.json")
    parser.add_argument("--val_json", default="data/raw/val.json")
    parser.add_argument("--num_classes", type=int, default=500)
    parser.add_argument("--train_per_class", type=int, default=40)
    parser.add_argument("--val_per_class", type=int, default=10)
    parser.add_argument("--test_per_class", type=int, default=10)
    parser.add_argument("--seed", type=int, default=500)
    parser.add_argument("--raw_root", default="data/raw")
    parser.add_argument("--output_dir", default="data/metadata")
    parser.add_argument("--class_selection_method", default="uniform_random_sampling")
    args = parser.parse_args()

    random.seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    print(f"Loading {args.train_json} ...")
    train_images, train_img2cat, categories = load_annotation_file(args.train_json)
    print(f"Loading {args.val_json} ...")
    val_images, val_img2cat, _ = load_annotation_file(args.val_json)

    train_grouped = group_images_by_category(train_images, train_img2cat)
    val_grouped = group_images_by_category(val_images, val_img2cat)

    # Only keep categories that have enough images in BOTH train_mini and official val
    eligible_categories = [
        cid for cid in categories
        if len(train_grouped.get(cid, [])) >= (args.train_per_class + args.val_per_class)
        and len(val_grouped.get(cid, [])) >= args.test_per_class
    ]
    print(f"{len(eligible_categories)} / {len(categories)} categories have enough images.")

    if len(eligible_categories) < args.num_classes:
        raise ValueError(
            f"Only {len(eligible_categories)} eligible categories found, "
            f"need {args.num_classes}. Check train_per_class/val_per_class/test_per_class."
        )

    selected_categories = sorted(random.sample(eligible_categories, args.num_classes))
    # class_idx assignment: shuffle once (seeded) so idx 0 isn't just "smallest category id"
    random.shuffle(selected_categories)

    class_to_idx = {str(cat_id): idx for idx, cat_id in enumerate(selected_categories)}
    idx_to_class = {
        str(idx): {
            "original_class_id": cat_id,
            "class_name": categories[cat_id]["name"],
            "category": categories[cat_id]["category"],
        }
        for idx, cat_id in enumerate(selected_categories)
    }

    # --- selected_classes.csv ---
    with open(os.path.join(args.output_dir, "selected_classes.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_idx", "original_class_id", "class_name", "category"])
        for idx, cat_id in enumerate(selected_categories):
            info = categories[cat_id]
            writer.writerow([idx, cat_id, info["name"], info["category"]])

    with open(os.path.join(args.output_dir, "class_to_idx.json"), "w") as f:
        json.dump(class_to_idx, f, indent=2)
    with open(os.path.join(args.output_dir, "idx_to_class.json"), "w") as f:
        json.dump(idx_to_class, f, indent=2)

    # --- train / val (from train_mini) + test (from official val) ---
    split_rows = {"train": [], "val": [], "test": []}
    global_image_id = 0  # guarantees uniqueness across train_mini + val, per Section 3.1's requirement

    for cat_id in selected_categories:
        idx = class_to_idx[str(cat_id)]
        class_name = categories[cat_id]["name"]

        train_pool = train_grouped[cat_id][:]
        random.shuffle(train_pool)
        train_ids = train_pool[: args.train_per_class]
        val_ids = train_pool[args.train_per_class: args.train_per_class + args.val_per_class]

        test_pool = val_grouped[cat_id][:]
        random.shuffle(test_pool)
        test_ids = test_pool[: args.test_per_class]

        for split_name, id_list, image_lookup in [
            ("train", train_ids, train_images),
            ("val", val_ids, train_images),
            ("test", test_ids, val_images),
        ]:
            for orig_img_id in id_list:
                path = normalize_path(args.raw_root, image_lookup[orig_img_id])
                split_rows[split_name].append([
                    global_image_id, path, cat_id, idx, class_name, split_name
                ])
                global_image_id += 1

    for split_name in ["train", "val", "test"]:
        out_path = os.path.join(args.output_dir, f"{split_name}.csv")
        with open(out_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["image_id", "image_path", "original_class_id", "class_idx", "class_name", "split"])
            writer.writerows(split_rows[split_name])
        print(f"Wrote {out_path} ({len(split_rows[split_name])} rows)")

    # --- split_config.json ---
    split_config = {
        "random_seed": args.seed,
        "num_classes": args.num_classes,
        "class_selection_method": args.class_selection_method,
        "train_images_per_class": args.train_per_class,
        "val_images_per_class": args.val_per_class,
        "test_images_per_class": args.test_per_class,
        "image_size": 224,
        "dataset": "iNaturalist-2021",
    }
    with open(os.path.join(args.output_dir, "split_config.json"), "w") as f:
        json.dump(split_config, f, indent=2)

    print("\nDone. All metadata files written to", args.output_dir)


if __name__ == "__main__":
    main()


Overwriting scripts/build_dataset.py


In [8]:
!python scripts/build_dataset.py \
    --train_json data/raw/train_mini.json \
    --val_json data/raw/val.json \
    --num_classes 500 \
    --seed 500 \
    --output_dir data/metadata


Loading data/raw/train_mini.json ...
Loading data/raw/val.json ...
10000 / 10000 categories have enough images.
Wrote data/metadata\train.csv (20000 rows)
Wrote data/metadata\val.csv (5000 rows)
Wrote data/metadata\test.csv (5000 rows)

Done. All metadata files written to data/metadata


### Sanity-check the output

In [9]:
import json, pandas as pd

split_config = json.load(open("data/metadata/split_config.json"))
print("split_config.json:", split_config)

train_df = pd.read_csv("data/metadata/train.csv")
print("\ntrain.csv shape:", train_df.shape)
print(train_df.head())
print("\nclass_idx range:", train_df["class_idx"].min(), "-", train_df["class_idx"].max())
print("num unique classes in train.csv:", train_df["class_idx"].nunique())


split_config.json: {'random_seed': 500, 'num_classes': 500, 'class_selection_method': 'uniform_random_sampling', 'train_images_per_class': 40, 'val_images_per_class': 10, 'test_images_per_class': 10, 'image_size': 224, 'dataset': 'iNaturalist-2021'}

train.csv shape: (20000, 6)
   image_id                                         image_path  \
0         0  data/raw/train_mini/06729_Plantae_Tracheophyta...   
1         1  data/raw/train_mini/06729_Plantae_Tracheophyta...   
2         2  data/raw/train_mini/06729_Plantae_Tracheophyta...   
3         3  data/raw/train_mini/06729_Plantae_Tracheophyta...   
4         4  data/raw/train_mini/06729_Plantae_Tracheophyta...   

   original_class_id  class_idx      class_name  split  
0               6729          0  Cota tinctoria  train  
1               6729          0  Cota tinctoria  train  
2               6729          0  Cota tinctoria  train  
3               6729          0  Cota tinctoria  train  
4               6729          0  Cota t

## 5. Build the long-tail experiment data (Section 9)

This is A's own experiment (due 7.25, run on top of C's training
pipeline once it exists). Produces `longtail_train.csv` and
`longtail_config.json` (and `longtail_resampled_train.csv` if you pass
`--static_oversample`).


In [10]:
%%writefile scripts/build_longtail.py
"""
scripts/build_longtail.py
----------------------------
Constructs the long-tail experiment data (Section 9 of the spec).

Step 1 (always): subsample the balanced train.csv into an exponentially
decaying long-tail distribution -> longtail_train.csv
Step 2 (only if --static_oversample): duplicate minority-class rows to
balance it -> longtail_resampled_train.csv. Otherwise (default,
recommended), leave resampling to get_dataloader(use_weighted_sampler=True)
at train time -- no second CSV needed.

Usage:
    # Method B (dynamic sampler, recommended -- only writes longtail_train.csv)
    python scripts/build_longtail.py --seed 500 --min_per_class 5 --max_per_class 40

    # Method A (static oversampling -- also writes longtail_resampled_train.csv)
    python scripts/build_longtail.py --seed 500 --min_per_class 5 --max_per_class 40 --static_oversample
"""

import os
import csv
import json
import random
import argparse
from collections import defaultdict


def load_train_csv(path):
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    return rows


def group_by_class(rows):
    grouped = defaultdict(list)
    for row in rows:
        grouped[int(row["class_idx"])].append(row)
    return grouped


def exponential_decay_counts(num_classes, min_per_class, max_per_class):
    """n_r = max_per_class * mu^r, chosen so n_(num_classes-1) = min_per_class.
    Returns a list of length num_classes (one target count per class rank)."""
    if num_classes == 1:
        return [max_per_class]
    mu = (min_per_class / max_per_class) ** (1.0 / (num_classes - 1))
    return [max(min_per_class, round(max_per_class * (mu ** r))) for r in range(num_classes)]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_csv", default="data/metadata/train.csv")
    parser.add_argument("--output_dir", default="data/metadata")
    parser.add_argument("--seed", type=int, default=500)
    parser.add_argument("--min_per_class", type=int, default=5)
    parser.add_argument("--max_per_class", type=int, default=40)
    parser.add_argument("--static_oversample", action="store_true",
                         help="Also write longtail_resampled_train.csv (方式A). "
                              "Default is 方式B: only write longtail_train.csv and "
                              "let get_dataloader(use_weighted_sampler=True) handle balancing.")
    args = parser.parse_args()

    random.seed(args.seed)
    rows = load_train_csv(args.train_csv)
    grouped = group_by_class(rows)

    class_indices = sorted(grouped.keys())
    num_classes = len(class_indices)

    # Randomize WHICH classes are "head" vs "tail" (don't just use class_idx order,
    # since class_idx assignment was already arbitrary/shuffled in build_dataset.py)
    class_order = class_indices[:]
    random.shuffle(class_order)

    target_counts = exponential_decay_counts(num_classes, args.min_per_class, args.max_per_class)

    longtail_rows = []
    per_class_kept = {}
    for rank, class_idx in enumerate(class_order):
        pool = grouped[class_idx][:]
        random.shuffle(pool)
        n_keep = min(target_counts[rank], len(pool))
        kept = pool[:n_keep]
        longtail_rows.extend(kept)
        per_class_kept[class_idx] = n_keep

    out_path = os.path.join(args.output_dir, "longtail_train.csv")
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image_id", "image_path", "original_class_id",
                                                "class_idx", "class_name", "split"])
        writer.writeheader()
        writer.writerows(longtail_rows)
    print(f"Wrote {out_path} ({len(longtail_rows)} rows, "
          f"{min(per_class_kept.values())}-{max(per_class_kept.values())} images/class)")

    resampling_strategy = "weighted_random_sampler"

    if args.static_oversample:
        # 方式A: duplicate rows in minority classes up to max_per_class.
        # NOTE: image_id is no longer unique in this file (rows repeat) -- per spec
        # Section 9, this file is only a training input list, not for general stats.
        resampled_rows = []
        for class_idx in class_order:
            kept = [r for r in longtail_rows if int(r["class_idx"]) == class_idx]
            resampled = kept[:]
            i = 0
            while len(resampled) < args.max_per_class:
                resampled.append(kept[i % len(kept)])
                i += 1
            resampled_rows.extend(resampled)

        resampled_path = os.path.join(args.output_dir, "longtail_resampled_train.csv")
        with open(resampled_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["image_id", "image_path", "original_class_id",
                                                    "class_idx", "class_name", "split"])
            writer.writeheader()
            writer.writerows(resampled_rows)
        print(f"Wrote {resampled_path} ({len(resampled_rows)} rows, static oversampling)")
        resampling_strategy = "static_oversampling"

    longtail_config = {
        "random_seed": args.seed,
        "longtail_ratio": "exponential_decay",
        "min_images_per_class": args.min_per_class,
        "max_images_per_class": args.max_per_class,
        "resampling_strategy": resampling_strategy,
    }
    config_path = os.path.join(args.output_dir, "longtail_config.json")
    with open(config_path, "w") as f:
        json.dump(longtail_config, f, indent=2)
    print(f"Wrote {config_path}")

    print("\nNext step: train two models for comparison --")
    print("  resnet18_scratch_longtail_unbalanced  <- train on longtail_train.csv directly")
    print("  resnet18_scratch_longtail_resampled   <- train on longtail_train.csv WITH")
    print("                                            use_weighted_sampler=True (format B)")
    print("                                            or on longtail_resampled_train.csv (format A)")


if __name__ == "__main__":
    main()


Overwriting scripts/build_longtail.py


In [11]:
!python scripts/build_longtail.py \
    --train_csv data/metadata/train.csv \
    --output_dir data/metadata \
    --seed 500 \
    --min_per_class 5 \
    --max_per_class 40
    # add --static_oversample if you want longtail_resampled_train.csv too (format A)


Wrote data/metadata\longtail_train.csv (8421 rows, 5-40 images/class)
Wrote data/metadata\longtail_config.json

Next step: train two models for comparison --
  resnet18_scratch_longtail_unbalanced  <- train on longtail_train.csv directly
  resnet18_scratch_longtail_resampled   <- train on longtail_train.csv WITH
                                            use_weighted_sampler=True (format B)
                                            or on longtail_resampled_train.csv (format A)


## 6. Extract the small subset to upload to Google Drive

Per the course forum confirmation: you may select classes on the full
42GB download locally, then only upload the ~2-3GB actually-used subset.
This copies only the images referenced in train/val/test.csv into
`data/subset/`, preserving the same relative path structure.


In [12]:
%%writefile scripts/extract_subset.py
"""
scripts/extract_subset.py
----------------------------
Per the course forum confirmation: you may select the 500 classes locally
on the FULL 42GB download, then only upload the small subset actually
used (should end up ~2-3GB) to Google Drive for Colab training.

This script reads train.csv / val.csv / test.csv (already generated by
build_dataset.py) and COPIES only the referenced image files into a new
"data/subset/" folder, preserving the same relative path structure so
image_path values in the CSVs still resolve correctly once you swap
data/raw -> data/subset (or just rename data/subset to data/raw on Drive).

Usage:
    python scripts/extract_subset.py \
        --metadata_dir data/metadata \
        --raw_root . \
        --output_dir data/subset

After running, upload BOTH:
    data/subset/           (the actual images, ~2-3GB)
    data/metadata/          (the CSVs/JSONs, a few MB)
to Google Drive. On Colab, either:
    (a) rename data/subset -> data/raw after upload, or
    (b) pass --raw_root pointing at wherever you put the subset folder
        when running get_dataloader() downstream (the image_path column
        already starts with "data/raw/...", so option (a) needs no code
        changes; option (b) would need path rewriting -- (a) is simpler).
"""

import os
import csv
import shutil
import argparse


def collect_referenced_paths(metadata_dir):
    """Reads train.csv/val.csv/test.csv and returns the set of unique image_path values."""
    paths = set()
    for split in ["train", "val", "test"]:
        csv_path = os.path.join(metadata_dir, f"{split}.csv")
        if not os.path.exists(csv_path):
            print(f"WARNING: {csv_path} not found, skipping.")
            continue
        with open(csv_path, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                paths.add(row["image_path"])
    return paths


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--metadata_dir", default="data/metadata")
    parser.add_argument("--raw_root", default=".",
                         help="Root to resolve image_path against (image_path already "
                              "starts with 'data/raw/...', so usually leave this as '.').")
    parser.add_argument("--output_dir", default="data/subset")
    parser.add_argument("--dry_run", action="store_true",
                         help="Just report how many files / how much size, don't copy.")
    args = parser.parse_args()

    referenced_paths = collect_referenced_paths(args.metadata_dir)
    print(f"{len(referenced_paths)} unique images referenced across train/val/test.csv")

    total_bytes = 0
    missing = []
    copied = 0

    for rel_path in sorted(referenced_paths):
        src_path = os.path.join(args.raw_root, rel_path)

        if not os.path.exists(src_path):
            missing.append(src_path)
            continue

        size = os.path.getsize(src_path)
        total_bytes += size

        if not args.dry_run:
            # Preserve the same relative structure, but rooted at output_dir instead
            # of "data/raw" -- e.g. "data/raw/train_mini/00001/x.jpg"
            #                    -> "data/subset/train_mini/00001/x.jpg"
            rel_under_raw = os.path.relpath(rel_path, "data/raw") if rel_path.startswith("data/raw") else rel_path
            dst_path = os.path.join(args.output_dir, rel_under_raw)
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy2(src_path, dst_path)
            copied += 1

    print(f"Total size of referenced images: {total_bytes / (1024**3):.2f} GB")
    if missing:
        print(f"WARNING: {len(missing)} referenced images were not found on disk "
              f"(first 5 shown):")
        for p in missing[:5]:
            print(f"  {p}")

    if args.dry_run:
        print("\n(dry run -- nothing copied. Re-run without --dry_run to actually copy.)")
    else:
        print(f"\nCopied {copied} images to {args.output_dir}")
        print(f"Next: upload {args.output_dir}/ and {args.metadata_dir}/ to Google Drive.")
        print(f"On Colab, rename/mount {args.output_dir} as data/raw so image_path values "
              f"in the CSVs resolve without any code changes.")


if __name__ == "__main__":
    main()


Overwriting scripts/extract_subset.py


In [13]:
# Dry run first to see the size before actually copying:
!python scripts/extract_subset.py --dry_run


30000 unique images referenced across train/val/test.csv
Total size of referenced images: 2.56 GB

(dry run -- nothing copied. Re-run without --dry_run to actually copy.)


In [14]:
!python scripts/extract_subset.py
# Next: upload data/subset/ (rename to data/raw on Drive) and data/metadata/ to Google Drive.


30000 unique images referenced across train/val/test.csv
Total size of referenced images: 2.56 GB

Copied 30000 images to data/subset
Next: upload data/subset/ and data/metadata/ to Google Drive.
On Colab, rename/mount data/subset as data/raw so image_path values in the CSVs resolve without any code changes.


## 7. Smoke test: load a real batch through the DataLoader

Confirms `src/data/dataloader.py` actually works end-to-end on real
images before handing off to B/C/D.


In [15]:
import sys
sys.path.insert(0, ".")
from src.data.dataloader import get_dataloader

train_loader = get_dataloader(split="train", transform_type="basic_aug", batch_size=8)
images, labels, image_ids = next(iter(train_loader))
print("images shape:", images.shape)   # expect [8, 3, 224, 224]
print("labels:", labels)
print("image_ids:", image_ids)


images shape: torch.Size([8, 3, 224, 224])
labels: tensor([194, 299, 168, 197, 222,  67, 293, 255])
image_ids: ('11648', '17940', '10100', '11828', '13333', '4033', '17591', '15322')
